In [0]:
%run /Workspace/Users/asifbrohi@hotmail.co.uk/data-bricks_example/notebooks/write_to_unity_catalog

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark
import time

In [0]:
orders=spark.sql("select * from dev.raw_uki_ecommerce.ecommerce_orders")
products=spark.sql("select * from dev.raw_uki_ecommerce.ecommerce_products")

# Create stg dataframes ready to ingest

In [0]:
# this block explodes and flattens all orders data into a single row per order 
# this block creates new columns that you can flatten to a single row per order
# dataframe is used as a lookup table to create stg tables 

exploded_df=orders.withColumn("customer_exploded",F.explode(F.array("customer"))) \
    .withColumn("items_exploded",F.explode("items")) \
        .withColumn("payment_exploded",F.explode(F.array("payment")))\
            .withColumn("shipping_exploded",F.explode(F.array("shipping"))) \
                .withColumn("gift_exploded",F.explode(F.array("gift"))) \
                    .withColumn("totals_exploded",F.explode(F.array("totals"))) \
                        .withColumn("metadata_exploded",F.explode(F.array("metadata"))) 

# display(exploded_df)


# Create stg tables 


In [0]:
def date_format(df,column_name:str):
    """
    This function takes a dataframe and a column name and returns a dataframe with the column formatted as a date
    Args:
        df (pyspark.sql.DataFrame): dataframe to format
        column_name (str): column name to format
    Returns:
        pyspark.sql.DataFrame: dataframe with the column formatted as a date"""

    col=df.withColumn(column_name,F.to_date(F.col(column_name),'yyyyMMdd'))
    return col


In [0]:
# create function to write to landing and unity catalog 
def write_to_delta_lake(df,mode,landing_file_name:str,params:dict):
    """
    This function takes a dataframe, landing file name , mode, params and updates delta table in unity catalog

    Args:
        df (pyspark.sql.DataFrame): dataframe to write to delta table
        mode (str): mode to write to delta table
        landing_file_name (str): landing file name
        params (dict): params to write to delta table
    
    Returns:
        None
    
    """

    try:
        df.write.mode(mode).format("parquet").save(f"/Volumes/dev/stg_uki_ecommerce/landing/{landing_file_name}.parquet")
        print("written to volume")
        print("==============================================")
        time.sleep(0.2)
        print("Setting up params....")
        print("==============================================")
        time.sleep(0.2)
        print("Writing to Unity Catalog...")
        print("==============================================")
        time.sleep(0.2)
        UnityCatalogWriter(params).write_to_unity_catalog()
        print("Thank you delta table updated")
        print("==============================================")

    except Exception as e:
        print(e)


In [0]:
# create stg order table 
orderstg=exploded_df.select(
    F.col("order_id"),
    F.col("order_date"),
    F.col("order_status"),
    F.col("channel"),
    F.col("gift_exploded.is_gift"),
    F.col("gift_exploded.gift_message"),
    F.col("totals_exploded.subtotal"),
    F.col("totals_exploded.tax"),
    F.col("totals_exploded.shipping_cost"),
    F.col("totals_exploded.promo_discount"),
    F.col("totals_exploded.grand_total"),
    F.col("customer_exploded.customer_id"),
    F.col("items_exploded.product_id"),
    F.col("payment_exploded.transaction_id"),
    F.col("shipping_exploded.tracking_id"),
    F.col("metadata_exploded.session_id")
)
orderstg=date_format(orderstg,'order_date')

# =========================================================================================
write_to_delta_lake(orderstg,"overwrite","orderstg",
                    params={
                    "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
                    "parquet_file_name": "orderstg",
                    "method":"overwrite",
                    "table_name":"orders",
                    "catalog_name":"dev",
                    "schema_name":"stg_uki_ecommerce"
                    })



In [0]:
"""

This cell creates a new dataframe from exploded dataframe customers and explodes and flattens the address column

Writes the dataframe into a parquet file in the landing volume 

Write to unity catalog using UnityCatalogWriter


"""


customers_sort=exploded_df.select(
    F.col("customer_exploded.*")

)
# flatten address column
customers_stg=customers_sort.withColumn("address_exploded",F.explode(F.array(F.col("address")))) \
    .select(
        F.col("customer_id"),
        F.col("name"),
        F.col("email"),
        F.col("address_exploded.*"),
        F.col("loyalty_tier")
    )
# ==================================================================
write_to_delta_lake(customers_stg,"overwrite","customer_stg",params={
    "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
    "parquet_file_name": "customer_stg",
    "method":"overwrite",
    "table_name":"customers",
    "catalog_name":"dev",
    "schema_name":"stg_uki_ecommerce"})


In [0]:

shipping=exploded_df.select(
    F.col("shipping_exploded.*")
    
)
stg_shipping=shipping.withColumn("address_exploded",F.explode(F.array(F.col("address"))))\
    .select(
        F.col("tracking_id"),
        F.col("carrier"),
        F.col("cost"),
        F.col("address_exploded.*"),
        F.col("status"),
        F.col("estimated_delivery_date")
    )

stg_shipping=date_format(stg_shipping,"estimated_delivery_date")

# ========================================================================

write_to_delta_lake(stg_shipping,"overwrite","shipping_stg",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "shipping_stg",
        "method":"overwrite",
        "table_name":"shippingitems",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })

In [0]:
stg_shipping_events=shipping.withColumn("delivery_address_exploded",F.explode(F.array(F.col("address"))))\
    .withColumn("events_exploded",F.explode(F.col("events")))\
        .select(
            F.col("tracking_id"),
            F.col("events_exploded.*")
        )
stg_shipping_events=date_format(stg_shipping_events,'event_date')
# ==================================================================================

write_to_delta_lake(stg_shipping_events,"overwrite","shipping_stg_events",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "shipping_stg_events",
        "method":"overwrite",
        "table_name":"shippingevents",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })

In [0]:
payment=exploded_df.select(
    F.col("payment_exploded.*")
)
stg_payment=payment.select(
    F.col("transaction_id"),
    F.col("method"),
    F.col("status"),
    F.col("installments"),
    F.col("promo_code"),
    F.col("promo_discount"),
    F.col("amount_paid")
)

# =====================================================================
write_to_delta_lake(stg_payment,"overwrite","payments_stg",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "payments_stg",
        "method":"overwrite",
        "table_name":"payments",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })

In [0]:
metadata=exploded_df.select(
    F.col("metadata_exploded.*")
)

stg_metadata=metadata.select(
    F.col("session_id"),
    F.col("device"),
    F.col("referrer")
)

# =====================================================================
write_to_delta_lake(stg_metadata,"overwrite","metadata_stg",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "metadata_stg",
        "method":"overwrite",
        "table_name":"metadata",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })


In [0]:
items=exploded_df.select(
    F.col("items_exploded.*")
)
stgorderitems=items.withColumn("review_exploded",F.explode(F.array(F.col("review"))))\
    .select(
        F.col("product_id"),
        F.col("product_name"),
        F.col("category"),
        F.col("brand"),
        F.col("quantity"),
        F.col("list_price"),
        F.col("discount_pct"),
        F.col("price_paid"),
        F.col("line_total"),
        F.col("review_exploded.*")
    )
# ================================================================================== 
write_to_delta_lake(stgorderitems,"overwrite","orderitems_stg",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "orderitems_stg",
        "method":"overwrite",
        "table_name":"orderitems",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })


In [0]:
products_exploded=products.withColumn("attributes_exploded",F.explode(F.array(F.col("attributes"))))\
    .withColumn("inventory_exploded",F.explode(F.array(F.col("inventory")))) \
        .withColumn("ratings_exploded",F.explode(F.array(F.col("ratings")))) \
            .withColumn("tags_exploded",F.explode(F.array(F.col("tags")))) \
                .withColumn("pricing_exploded",F.explode(F.array(F.col("pricing")))) \
                    .withColumn("supplier_exploded",F.explode(F.array(F.col("supplier"))))

display(products_exploded)


In [0]:
"""products stg table"""

stgproducts=products_exploded.select(
    F.col("product_id"),
    F.col("name"),
    F.col("category"),
    F.col("brand"),
    F.col("pricing_exploded.*"),
    F.col("created_date"),
    F.col("supplier_exploded.supplier_id")
)
stg_products=date_format(stgproducts,"created_date")
write_to_delta_lake(stg_products,"overwrite","products_stg",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "products_stg",
        "method":"overwrite",
        "table_name":"products",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })


In [0]:
inventory=products_exploded.select("product_id","inventory_exploded.*")
stginventory=inventory.withColumn("warehouse_stock_exploded",F.explode(F.col("warehouse_stock")))\
    .select(
        F.col("product_id"),
        F.col("warehouse_stock_exploded.*"),
        F.col("reorder_level")
    )
# =============================================================================
write_to_delta_lake(stginventory,"overwrite","inventory_stg",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "inventory_stg",
        "method":"overwrite",
        "table_name":"inventory",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })


In [0]:
supplier_stg=products_exploded.select(F.col("supplier_exploded.*")).distinct()

# =======================================================================
write_to_delta_lake(stginventory,"overwrite","supplier_stg",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "supplier_stg",
        "method":"overwrite",
        "table_name":"suppliers",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })  

In [0]:
# display(products_exploded)
product_attribute=products_exploded.select(F.col("product_id"),F.col("attributes_exploded.*"))
product_attribute_stg=product_attribute.withColumn("color",F.explode(F.col("available_colors")))\
    .withColumn("size",F.explode(F.col("available_sizes")))\
    .select(
        F.col("product_id"),
        F.col("color"),
        F.col("size"),
        F.col("material")
    )
# =============================================================================================
write_to_delta_lake(product_attribute_stg,"overwrite","product_attribute_stg",params={
        "volume_path": "/Volumes/dev/stg_uki_ecommerce/landing",
        "parquet_file_name": "product_attribute_stg",
        "method":"overwrite",
        "table_name":"product_attributes",
        "catalog_name":"dev",
        "schema_name":"stg_uki_ecommerce"
        })  